# NB 42 - Reach-targeting features + observability smoke (gpt-5-mini)

**Purpose.** End-to-end verification of the two new reach-targeting features and the new output
artefacts they feed, on small OpenAI `gpt-5-mini` runs.

- **Feature 1 - persuadable targeting.** With `reach_a<1.0` and `reach_targeting_a="persuadable"`,
  each side keeps the most *undecided* audience members (smallest `|ground-truth package index|`).
- **Feature 2 - centrality targeting.** `reach_targeting_a="degree"` (on a Barabasi-Albert graph)
  keeps the most *central* audience members.
- **New outputs (byproduct).** `agent_prompts.csv` (full system+user+response for a stratified
  sample of agents), `reached_by_a/_b` flags in `agent_attributes.csv`, `bucket_summary.csv`,
  `targeting_diagnostics.csv`, plus the new O2 figures.

**Model / canon.** `gpt-5-mini` (OpenAI) for both messaging and surveys, `thinking=False`,
`day0_anchor=ground_truth_with_rationale`, and the production memory canon
`memory={"day0_anchor":{"ttl_days":1}}` (Day-0 anchor trails off after Day 1) - matching the
`tier1` AIRE runs. The API key is read automatically from `data/api_key.csv` (repo root) by the
simulation; the setup cell also loads it for a display sanity check.

**Caveat.** At `n=24` the committed-minority buckets (A-only / B-only) are ~1 agent each, so
per-bucket numbers there are noisy. This NB checks *plumbing*, not effect sizes.

**Cost.** ~24 agents x 3 days x 6 policies + anchoring + surveys + reflections + peer messaging is
a few hundred `gpt-5-mini` calls per run (x2 runs) - a few minutes and pennies of API cost.

## 1. Setup + local-server ping

In [1]:
import os, sys, json, logging
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', force=True)
for noisy in ('httpx', 'openai', 'httpcore'):
    logging.getLogger(noisy).setLevel(logging.WARNING)

import numpy as np
import pandas as pd

from cag.io.survey import load
from cag.io.llm import load_api_key
from cag.__main__ import build_nation
from cag.abm.sim import run_simulation, SIM_CONFIG
from cag.io.results import _write_all_csvs, _serialise_config, _atomic_write_json
from cag.io.plots import save_result_plots

PROVIDER = 'openai'
MODEL    = 'gpt-5-mini'

# The sim resolves the key itself from <repo>/data/api_key.csv; load here just for a sanity check.
API_KEY = load_api_key(PROVIDER, csv_path='../data/api_key.csv')
print(f'Provider/model : {PROVIDER}/{MODEL}')
print(f'API key loaded : {bool(API_KEY)} (len={len(API_KEY) if API_KEY else 0})')

Provider/model : openai/gpt-5-mini
API key loaded : True (len=164)


## 2. Config + run helper

We start from a copy of the current `SIM_CONFIG` (so every default is inherited, including the new
`reach_targeting_a/_b='random'`) and override only the smoke knobs. Crucially we pin
`memory={'day0_anchor':{'ttl_days':1}}` to match the production / `tier1` canon.

In [2]:
N_CITIZENS  = 24
RANDOM_SEED = 42
YEAR        = 2026

# 3 alternating package-mode days; P-A/P-B order flips each day; C (peer) always last.
DAYS = [
    {'phases': ['P-A', 'P-B', 'C']},
    {'phases': ['P-B', 'P-A', 'C']},
    {'phases': ['P-A', 'P-B', 'C']},
]

DATA = load('../data/yougov_survey_data/YouGovProcessedData.csv')


def make_config(**overrides):
    cfg = dict(SIM_CONFIG)
    cfg.update(
        n_citizens=N_CITIZENS,
        days=DAYS,
        k_peers_per_day=2,
        communication_mode='package',
        day0_anchor='ground_truth_with_rationale',
        memory={'day0_anchor': {'ttl_days': 1}},   # production canon: anchor trails off after Day 1
        thinking=False,
        llm_provider=PROVIDER,       # openai
        llm_model=MODEL,             # gpt-5-mini
        random_seed=RANDOM_SEED,
    )
    cfg.update(overrides)
    return cfg


def run_and_save(overrides, outdir):
    """Fresh nation -> run -> write CSVs + plots + config.json to a FIXED dir."""
    data = DATA.sample(n=N_CITIZENS, random_state=RANDOM_SEED).reset_index(drop=True)
    nation = build_nation(data, year=YEAR)
    cfg = make_config(**overrides)
    results = run_simulation(cfg, nation)
    out = Path(outdir)
    out.mkdir(parents=True, exist_ok=True)
    _write_all_csvs(out, results)
    save_result_plots(results, out)
    _atomic_write_json(_serialise_config(results['config']), out / 'config.json')
    print(f'\nSaved run to {out}')
    return out, results


print('reach_targeting defaults in SIM_CONFIG:',
      SIM_CONFIG.get('reach_targeting_a'), SIM_CONFIG.get('reach_targeting_b'))
print(f'n={N_CITIZENS}, days={len(DAYS)}, memory=ttl_days:1, thinking=False, model={MODEL}')

2026-07-05 20:54:10,995 [INFO] Columns with NaN counts (before filtering):
tprofile_gross_household    398
Political_Left_Right          6
dtype: int64
2026-07-05 20:54:10,996 [INFO] Column with most NaNs: tprofile_gross_household (398 NaNs)
2026-07-05 20:54:11,002 [INFO] 1483 rows after filtering.


reach_targeting defaults in SIM_CONFIG: random random
n=24, days=3, memory=ttl_days:1, thinking=False, model=gpt-5-mini


## 3. Run A - persuadable targeting (stochastic-block network)

Throttle side A to `reach_a=0.4` and target the **undecided**. Moderate reach so the reached set is
a strict subset of the audience (targeting has something to bite on).

In [3]:
OUT_A, res_a = run_and_save(
    {'reach_a': 0.4, 'reach_targeting_a': 'persuadable'},
    '../data/output/smoke/nb42_persuadable',
)

2026-07-05 20:55:13,054 [INFO] ===============================================================================
2026-07-05 20:55:13,054 [INFO]                             EXPERIMENT CONFIG
2026-07-05 20:55:13,055 [INFO] ===============================================================================
2026-07-05 20:55:13,056 [INFO] [run-shape]  n_citizens=24  days=3  seed=42  communication_mode=package
2026-07-05 20:55:13,056 [INFO] [exposure]   mode=rule_affinity_rank  targets=None  weights=None  audience_cap=None
2026-07-05 20:55:13,057 [INFO] [broadcast]  reach_a=0.40  reach_b=1.00  targeting_a=persuadable  targeting_b=random  message_source=offline  message_set=v1
2026-07-05 20:55:13,058 [INFO] [peer]       k_peers_per_day=2  network_type=stochastic_block  params={p_intra=0.15, p_inter=0.05}
2026-07-05 20:55:13,058 [INFO] [day0]       anchor=ground_truth_with_rationale
2026-07-05 20:55:13,059 [INFO] [memory]     preset=custom  verbatim_window_days=2  anchor=on(ttl=1)  own_reasoning=on 


Saved run to ../data/output/smoke/nb42_persuadable


### Verify Feature 1 - persuadable kept the undecided

`targeting_diagnostics.csv` compares the reached audience vs the dropped audience on mean
`|ground-truth package index|`. Persuadable targeting should keep the *smaller* `|GT|`.

In [4]:
td = pd.read_csv(OUT_A / 'targeting_diagnostics.csv')
display(td)

row = td[td['side'] == 'A'].iloc[0]
print(f"side A: targeting={row['targeting_mode']}, reach={row['reach']}, "
      f"reached |GT|={row['reached_mean_absgt']:.3f} vs dropped |GT|={row['dropped_mean_absgt']:.3f}")
assert row['reached_mean_absgt'] < row['dropped_mean_absgt'], \
    'FAIL: persuadable did not keep the more-undecided audience'
print('PASS: persuadable targeting kept the undecided (lower |GT|).')

attrs_a = pd.read_csv(OUT_A / 'agent_attributes.csv')
print('\nReached-by counts per bucket:')
print(attrs_a.groupby('political_exposure')[['reached_by_a', 'reached_by_b']].sum())

,side,targeting_mode,reach,n_audience,n_reached,n_dropped,reached_mean_absgt,dropped_mean_absgt
0,A,persuadable,0.4,15,6,9,0.888889,2.037037
1,B,random,1.0,15,15,0,1.444444,NaN


side A: targeting=persuadable, reach=0.4, reached |GT|=0.889 vs dropped |GT|=2.037
PASS: persuadable targeting kept the undecided (lower |GT|).

Reached-by counts per bucket:
                    reached_by_a  reached_by_b
political_exposure                            
A-only                         0             0
B-only                         0             1
both                           6            14
neither                        0             0


## 4. Run B - degree centrality targeting (Barabasi-Albert network)

BA has hubs, so degree centrality varies. Throttle side A to `reach_a=0.4` and target the
highest-degree audience members.

In [5]:
OUT_B, res_b = run_and_save(
    {'reach_a': 0.4, 'reach_targeting_a': 'degree',
     'network_type': 'barabasi_albert', 'network_params': {'m': 3}},
    '../data/output/smoke/nb42_degree',
)

2026-07-05 21:45:16,806 [INFO] ===============================================================================
2026-07-05 21:45:16,807 [INFO]                             EXPERIMENT CONFIG
2026-07-05 21:45:16,808 [INFO] ===============================================================================
2026-07-05 21:45:16,809 [INFO] [run-shape]  n_citizens=24  days=3  seed=42  communication_mode=package
2026-07-05 21:45:16,809 [INFO] [exposure]   mode=rule_affinity_rank  targets=None  weights=None  audience_cap=None
2026-07-05 21:45:16,810 [INFO] [broadcast]  reach_a=0.40  reach_b=1.00  targeting_a=degree  targeting_b=random  message_source=offline  message_set=v1
2026-07-05 21:45:16,810 [INFO] [peer]       k_peers_per_day=2  network_type=barabasi_albert  params={m=3}
2026-07-05 21:45:16,811 [INFO] [day0]       anchor=ground_truth_with_rationale
2026-07-05 21:45:16,812 [INFO] [memory]     preset=custom  verbatim_window_days=2  anchor=on(ttl=1)  own_reasoning=on  daily_summaries=on  reflecti


Saved run to ../data/output/smoke/nb42_degree


### Verify Feature 2 - degree kept the hubs

`targeting_diagnostics.csv` only tracks `|GT|` (which does *not* discriminate centrality), so we
verify directly: reconstruct the peer graph from `network_snapshot.json`, then compare the mean
degree of the reached vs dropped members of side A's audience.

In [6]:
import networkx as nx

snap = json.loads((OUT_B / 'network_snapshot.json').read_text())
G = nx.Graph()
for node in snap['nodes']:
    G.add_node(node['id'])
for u, v in snap['edges']:
    G.add_edge(u, v)
deg = {str(k): v for k, v in dict(G.degree()).items()}

attrs_b = pd.read_csv(OUT_B / 'agent_attributes.csv')
aud = attrs_b[attrs_b['political_exposure'].isin(['A-only', 'both'])].copy()
aud['degree'] = aud['agent_id'].astype(str).map(deg)
reached = aud[aud['reached_by_a'] == True]
dropped = aud[aud['reached_by_a'] == False]
mk, md = reached['degree'].mean(), dropped['degree'].mean()
print(f'side A audience n={len(aud)}: reached mean degree={mk:.2f} vs dropped mean degree={md:.2f}')
assert mk > md, 'FAIL: degree targeting did not keep the higher-degree (hub) audience'
print('PASS: degree targeting kept the more-central audience.')

side A audience n=15: reached mean degree=9.17 vs dropped mean degree=3.67
PASS: degree targeting kept the more-central audience.


## 5. New-output sanity checks (byproduct of the runs)

`agent_prompts.csv` should hold the full prompt trail for the stratified sample (~1 agent per
bucket), across every persona-facing stage. `bucket_summary.csv` is the one-row-per-bucket
cross-run seed.

In [7]:
ap = pd.read_csv(OUT_A / 'agent_prompts.csv')
print('agent_prompts.csv rows:', len(ap))
print('sampled agents:', ap['agent_id'].nunique(), '->', sorted(ap['agent_id'].unique()))
print('stages captured:', sorted(ap['stage'].unique()))
expected = {'seed_rationale', 'survey_reasoning', 'survey_answer',
            'reflection_broadcast', 'reflection_peer', 'peer_message'}
missing = expected - set(ap['stage'].unique())
print('missing stages:', missing or 'none')

one = ap[ap['agent_id'] == ap['agent_id'].iloc[0]].sort_values('prompt_seq')
print(f"\nPrompt sequence for agent {one['agent_id'].iloc[0]}:")
display(one[['prompt_seq', 'day', 'phase', 'stage', 'policy_id']].head(30))

print('\nbucket_summary.csv:')
display(pd.read_csv(OUT_A / 'bucket_summary.csv'))

agent_prompts.csv rows: 200
sampled agents: 4 -> [np.float64(103.0), np.float64(355.0), np.float64(620.0), np.float64(1001.0)]
stages captured: ['peer_message', 'reflection_broadcast', 'reflection_peer', 'seed_rationale', 'survey_answer', 'survey_reasoning']
missing stages: none

Prompt sequence for agent 103.0:


,prompt_seq,day,phase,stage,policy_id
0,0,0,NaN,seed_rationale,ClimatePolicyID(1)
1,1,0,NaN,seed_rationale,ClimatePolicyID(2)
2,2,0,NaN,seed_rationale,ClimatePolicyID(3)
3,3,0,NaN,seed_rationale,ClimatePolicyID(4)
4,4,0,NaN,seed_rationale,ClimatePolicyID(5)
5,5,0,NaN,seed_rationale,ClimatePolicyID(6)
6,6,1,C,peer_message,climate_policy_package
7,7,1,C,reflection_peer,climate_policy_package
8,8,1,survey,survey_reasoning,ClimatePolicyID(3)
9,9,1,survey,survey_answer,ClimatePolicyID(3)



bucket_summary.csv:


,run_label,political_exposure,n_agents,day0_gt_mean,end_mean,drift,mae,rank_rho,reached_a,reached_b
0,NaN,A-only,1,2.666667,2.666667,-3.333338e-10,3.333338e-10,NaN,0,0
1,NaN,B-only,1,-0.666667,-0.333333,3.333333e-01,3.333333e-01,NaN,0,1
2,NaN,both,14,1.023810,1.273810,2.500000e-01,2.500000e-01,0.924375,6,14
3,NaN,neither,8,0.020833,0.708333,6.875000e-01,6.875000e-01,0.506211,0,0
4,NaN,TOTAL,24,0.687500,1.076389,3.888889e-01,3.888889e-01,0.950810,6,15


## 6. Summary

If all three asserts passed (persuadable `|GT|`, degree hubs, stages captured), the two features and
the new output artefacts are wired correctly end-to-end. **NB 43** loads `nb42_persuadable` for the
full prompt walkthrough and the O2 figure gallery.